# 作业 3
2026年6月2日

## 2 卷积和池化层

In [1]:
# 2.1.1 计算卷积层输出 Feature Map 的尺寸
# 输入: 3 × 32 × 32 (C × H × W)
# 卷积核: 16 个, 每个 3 × 5 × 5, padding=2, stride=2

C_in, H_in, W_in = 3, 32, 32
num_kernels = 16
K_h, K_w = 5, 5
padding = 2
stride = 2

H_out = (H_in + 2 * padding - K_h) // stride + 1
W_out = (W_in + 2 * padding - K_w) // stride + 1
C_out = num_kernels

print(f"输出 Feature Map 尺寸: {C_out} × {H_out} × {W_out} (C × H × W)")
print(f"计算过程: H_out = (32 + 2×2 - 5) / 2 + 1 = {H_out}")
print(f"         W_out = (32 + 2×2 - 5) / 2 + 1 = {W_out}")

输出 Feature Map 尺寸: 16 × 16 × 16 (C × H × W)
计算过程: H_out = (32 + 2×2 - 5) / 2 + 1 = 16
         W_out = (32 + 2×2 - 5) / 2 + 1 = 16


In [2]:
# 2.1.2 计算单个输出通道中单个像素值所需的点积（乘法）运算次数
# 每个卷积核大小为 3 × 5 × 5，与输入对应区域做点积

C_kernel, K_h, K_w = 3, 5, 5
num_multiplications = C_kernel * K_h * K_w

print(f"单个输出通道单个像素所需的点积运算次数: {num_multiplications}")
print(f"计算过程: 3 × 5 × 5 = {num_multiplications}")

单个输出通道单个像素所需的点积运算次数: 75
计算过程: 3 × 5 × 5 = 75


In [3]:
# 2.2 手动实现 2D Max Pooling 前向传播（支持 stride 和 padding）
import numpy as np


def max_pool2d(X, pool_size, stride=None, padding=0):
    """2D 最大池化前向传播。

    参数:
        X: 输入数组，形状为 (N, C, H, W)
        pool_size: 池化窗口大小
        stride: 步长，默认为 pool_size
        padding: 零填充大小
    返回:
        池化后的数组
    """
    if stride is None:
        stride = pool_size

    X = np.asarray(X, dtype=np.float64)
    if X.ndim == 3:
        X = X[np.newaxis, ...]

    N, C, H, W = X.shape

    if padding > 0:
        X = np.pad(
            X,
            ((0, 0), (0, 0), (padding, padding), (padding, padding)),
            mode="constant",
            constant_values=-np.inf,
        )
        H += 2 * padding
        W += 2 * padding

    out_h = (H - pool_size) // stride + 1
    out_w = (W - pool_size) // stride + 1
    Y = np.zeros((N, C, out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            h_start = i * stride
            w_start = j * stride
            window = X[:, :, h_start : h_start + pool_size, w_start : w_start + pool_size]
            Y[:, :, i, j] = np.max(window, axis=(2, 3))

    return Y


# 测试
X = np.arange(16, dtype=np.float64).reshape(1, 1, 4, 4)
print("输入:\n", X[0, 0])
print("\nMaxPool2d(pool_size=2, stride=2):\n", max_pool2d(X, pool_size=2, stride=2)[0, 0])
print("\nMaxPool2d(pool_size=2, stride=1, padding=1):\n", max_pool2d(X, pool_size=2, stride=1, padding=1)[0, 0])

输入:
 [[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]
 [12. 13. 14. 15.]]

MaxPool2d(pool_size=2, stride=2):
 [[ 5.  7.]
 [13. 15.]]

MaxPool2d(pool_size=2, stride=1, padding=1):
 [[ 0.  1.  2.  3.  3.]
 [ 4.  5.  6.  7.  7.]
 [ 8.  9. 10. 11. 11.]
 [12. 13. 14. 15. 15.]
 [12. 13. 14. 15. 15.]]


## 3 LeNet, AlexNet, VGG 和 NiN

In [4]:
# 3.1.1 单个 5×5 卷积层（无偏置）的参数量
# 输入输出通道数均为 C

C = 64  # 示例通道数
params_5x5 = C * C * 5 * 5

print(f"单个 5×5 卷积层参数量（无偏置）: {C} × {C} × 5 × 5 = {params_5x5}")
print(f"通式: 25C²")

单个 5×5 卷积层参数量（无偏置）: 64 × 64 × 5 × 5 = 102400
通式: 25C²


In [5]:
# 3.1.2 两个串联 3×3 卷积层（无偏置）的总参数量
# 两层输入输出通道数均为 C

C = 64  # 示例通道数
params_3x3_one = C * C * 3 * 3
params_3x3_two = 2 * params_3x3_one

print(f"第一个 3×3 卷积层: {C} × {C} × 3 × 3 = {params_3x3_one}")
print(f"第二个 3×3 卷积层: {C} × {C} × 3 × 3 = {params_3x3_one}")
print(f"总参数量: {params_3x3_two}")
print(f"通式: 18C²（相比 5×5 的 25C² 减少了 28% 参数量）")

第一个 3×3 卷积层: 64 × 64 × 3 × 3 = 36864
第二个 3×3 卷积层: 64 × 64 × 3 × 3 = 36864
总参数量: 73728
通式: 18C²（相比 5×5 的 25C² 减少了 28% 参数量）


In [6]:
# 3.2 使用 torch.nn.Sequential 定义标准 NiN Block
import torch
from torch import nn


def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
    )


block = nin_block(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0)
x = torch.randn(1, 3, 224, 224)
y = block(x)
print(block)
print(f"\n输入形状: {x.shape}")
print(f"输出形状: {y.shape}")

Sequential(
  (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
  (1): ReLU()
  (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)

输入形状: torch.Size([1, 3, 224, 224])
输出形状: torch.Size([1, 96, 54, 54])


## 4 Inception, 批量归一化和残差网络

In [7]:
# 4.1 手动计算 Batch Normalization 输出
import math

x = [2, 4, 6, 8]
gamma = 2
beta = 1
eps = 0

mean = sum(x) / len(x)
var = sum((xi - mean) ** 2 for xi in x) / len(x)
std = math.sqrt(var + eps)

x_hat = [(xi - mean) / std for xi in x]
y = [gamma * xh + beta for xh in x_hat]

print(f"均值 μ = {mean}")
print(f"方差 σ² = {var}")
print(f"标准差 σ = {std:.4f}")
print(f"\n归一化值 x̂ = {[round(xh, 4) for xh in x_hat]}")
print(f"\n最终输出:")
for i, yi in enumerate(y, 1):
    print(f"  y_{i} = {yi:.4f}")

均值 μ = 5.0
方差 σ² = 5.0
标准差 σ = 2.2361

归一化值 x̂ = [-1.3416, -0.4472, 0.4472, 1.3416]

最终输出:
  y_1 = -1.6833
  y_2 = 0.1056
  y_3 = 1.8944
  y_4 = 3.6833


In [8]:
# 4.2 定义 Residual 残差块
import torch
from torch import nn


class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None

    def forward(self, X):
        Y = self.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3 is not None:
            X = self.conv3(X)
        Y += X
        return self.relu(Y)


block1 = Residual(3, 3, use_1x1conv=False)
block2 = Residual(3, 6, use_1x1conv=True, stride=2)
x = torch.randn(1, 3, 32, 32)
print("use_1x1conv=False:", block1(x).shape)
print("use_1x1conv=True, stride=2:", block2(x).shape)

use_1x1conv=False: torch.Size([1, 3, 32, 32])
use_1x1conv=True, stride=2: torch.Size([1, 6, 16, 16])


## 5 图像增广，微调和样式迁移

In [9]:
# 5.1.1 为什么对底层特征提取层设置较小学习率，对顶层输出层设置较大学习率？

answer = """
预训练模型的底层卷积层已经在大规模数据集（如 ImageNet）上学到了通用的、可迁移的低级和中级视觉特征
（如边缘、纹理、形状等），这些特征对大多数视觉任务都有用。若使用较大学习率，会破坏这些已学好的
通用特征，导致性能下降（灾难性遗忘）。

而顶层输出层（分类头）是随机初始化、专门为目标任务设计的，需要从头学习如何将提取到的特征映射到
目标类别的决策边界，因此需要较大的学习率以快速收敛。

简言之：底层参数已接近最优，只需微调；顶层参数需大幅更新。
"""
print(answer)


预训练模型的底层卷积层已经在大规模数据集（如 ImageNet）上学到了通用的、可迁移的低级和中级视觉特征
（如边缘、纹理、形状等），这些特征对大多数视觉任务都有用。若使用较大学习率，会破坏这些已学好的
通用特征，导致性能下降（灾难性遗忘）。

而顶层输出层（分类头）是随机初始化、专门为目标任务设计的，需要从头学习如何将提取到的特征映射到
目标类别的决策边界，因此需要较大的学习率以快速收敛。

简言之：底层参数已接近最优，只需微调；顶层参数需大幅更新。



In [10]:
# 5.1.2 目标数据集很小且与源数据集非常相似时，应采取什么微调策略防止过拟合？

answer = """
当目标数据集很小且与源数据集非常相似时，推荐以下微调策略：

1. 冻结预训练模型的全部或大部分底层特征提取层，只训练顶层分类器（线性探测），
   可最大程度保留预训练特征，避免在小数据集上过拟合。
2. 若需进一步微调，仅解冻靠近输出层的少数高层，并使用很小的学习率。
3. 加强数据增广（随机裁剪、翻转、颜色抖动等），增加有效训练样本多样性。
4. 使用早停（Early Stopping）、权重衰减（Weight Decay）等正则化手段。
5. 减小 batch size 或使用 Dropout 进一步抑制过拟合。

核心思路：数据少且相似 → 预训练特征已足够好 → 尽量冻结底层，只微调顶层，并加强正则化。
"""
print(answer)


当目标数据集很小且与源数据集非常相似时，推荐以下微调策略：

1. 冻结预训练模型的全部或大部分底层特征提取层，只训练顶层分类器（线性探测），
   可最大程度保留预训练特征，避免在小数据集上过拟合。
2. 若需进一步微调，仅解冻靠近输出层的少数高层，并使用很小的学习率。
3. 加强数据增广（随机裁剪、翻转、颜色抖动等），增加有效训练样本多样性。
4. 使用早停（Early Stopping）、权重衰减（Weight Decay）等正则化手段。
5. 减小 batch size 或使用 Dropout 进一步抑制过拟合。

核心思路：数据少且相似 → 预训练特征已足够好 → 尽量冻结底层，只微调顶层，并加强正则化。



In [11]:
# 5.2 使用 torchvision.transforms 构建图像增广流水线
from torchvision import transforms
from PIL import Image
import torch

transform = transforms.Compose([
    # 1. 随机裁剪（面积比例 0.08~1.0）并缩放到 224×224
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机改变亮度、对比度、饱和度，变化范围 0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 4. 转换为 PyTorch Tensor
    transforms.ToTensor(),
])

print(transform)

# 演示
dummy_img = Image.new("RGB", (256, 256), color=(128, 64, 32))
tensor = transform(dummy_img)
print(f"\n输出 Tensor 形状: {tensor.shape}")
print(f"数值范围: [{tensor.min():.4f}, {tensor.max():.4f}]")

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)

输出 Tensor 形状: torch.Size([3, 224, 224])
数值范围: [0.1137, 0.2745]


## 6 目标检测，计算机视觉训练技巧

In [12]:
# 6.1 计算两个边界框的 IoU
# 格式: [top-left x, top-left y, bottom-right x, bottom-right y]

A = [10, 10, 50, 50]  # Ground Truth
B = [30, 30, 70, 70]  # Prediction

inter_x1 = max(A[0], B[0])
inter_y1 = max(A[1], B[1])
inter_x2 = min(A[2], B[2])
inter_y2 = min(A[3], B[3])

inter_w = max(0, inter_x2 - inter_x1)
inter_h = max(0, inter_y2 - inter_y1)
inter_area = inter_w * inter_h

area_A = (A[2] - A[0]) * (A[3] - A[1])
area_B = (B[2] - B[0]) * (B[3] - B[1])
union_area = area_A + area_B - inter_area

iou = inter_area / union_area

print(f"Ground Truth A: {A}, 面积 = {area_A}")
print(f"Prediction B:   {B}, 面积 = {area_B}")
print(f"交集区域: [{inter_x1}, {inter_y1}, {inter_x2}, {inter_y2}], 面积 = {inter_area}")
print(f"并集面积 = {union_area}")
print(f"\nIoU = {inter_area} / {union_area} = {iou:.4f}")

Ground Truth A: [10, 10, 50, 50], 面积 = 1600
Prediction B:   [30, 30, 70, 70], 面积 = 1600
交集区域: [30, 30, 50, 50], 面积 = 400
并集面积 = 2800

IoU = 400 / 2800 = 0.1429


In [13]:
# 6.2 实现带 Label Smoothing 的交叉熵损失
import torch
import torch.nn.functional as F


def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, num_classes=None):
    """计算应用 Label Smoothing 后的交叉熵损失。

    参数:
        logits: 模型输出, 形状 (N, K)
        labels: 真实标签, 形状 (N,), 整数类别索引
        epsilon: 平滑因子
        num_classes: 类别数 K, 默认从 logits 推断
    返回:
        标量损失值
    """
    N, K = logits.shape
    if num_classes is not None:
        K = num_classes

    # 构造平滑后的目标分布
    smooth_labels = torch.full((N, K), epsilon / (K - 1), device=logits.device, dtype=logits.dtype)
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)

    log_probs = F.log_softmax(logits, dim=1)
    loss = -(smooth_labels * log_probs).sum(dim=1).mean()
    return loss


# 测试: K=5 分类, epsilon=0.1
torch.manual_seed(42)
logits = torch.randn(4, 5)
labels = torch.tensor([0, 2, 1, 4])
epsilon = 0.1
K = 5

loss = label_smoothing_cross_entropy(logits, labels, epsilon=epsilon, num_classes=K)
print(f"Label Smoothing 交叉熵损失 (ε={epsilon}, K={K}): {loss.item():.4f}")

# 展示平滑后的目标分布（以第一个样本 label=0 为例）
smooth_target = torch.full((K,), epsilon / (K - 1))
smooth_target[0] = 1.0 - epsilon
print(f"\n样本 0 的平滑目标分布: {smooth_target.tolist()}")

Label Smoothing 交叉熵损失 (ε=0.1, K=5): 1.4089

样本 0 的平滑目标分布: [0.8999999761581421, 0.02500000037252903, 0.02500000037252903, 0.02500000037252903, 0.02500000037252903]
